[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nabin2004/Machine-Learning-Bootcamp/blob/main/Module_05_Classification/04_random_forest.ipynb)

# Episode 14 – Random Forest & Ensemble Methods

**Machine Learning Bootcamp** | Module 05

---

## 🎯 Learning Objectives
- Understand bagging and the Random Forest algorithm
- Train and evaluate a Random Forest classifier
- Interpret feature importances from the ensemble

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import accuracy_score
from sklearn.datasets import load_breast_cancer

sns.set_theme(style='whitegrid')

## 1. Why Ensembles?

> *"The wisdom of crowds"* — combining many weak learners creates a strong learner.

**Bagging (Bootstrap Aggregation):**
1. Draw $B$ bootstrap samples from the training data.
2. Train a decision tree on each sample.
3. Predict by majority vote.

**Random Forest adds:** random feature selection at each split → decorrelates the trees.

In [ ]:
data = load_breast_cancer(as_frame=True)
X, y = data.data, data.target

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

models = {
    'Single Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest (100)':  RandomForestClassifier(n_estimators=100, random_state=42),
    'Gradient Boosting':    GradientBoostingClassifier(n_estimators=100, random_state=42),
}

results = {}
for name, m in models.items():
    m.fit(X_train, y_train)
    acc = accuracy_score(y_test, m.predict(X_test))
    cv  = cross_val_score(m, X, y, cv=5).mean()
    results[name] = {'Test Acc': acc, 'CV Acc': cv}
    print(f'{name:<30}  Test={acc:.4f}  CV={cv:.4f}')

pd.DataFrame(results).T

In [ ]:
# Feature importances from Random Forest
rf = models['Random Forest (100)']
importances = pd.Series(rf.feature_importances_, index=data.feature_names).sort_values(ascending=False)[:15]

plt.figure(figsize=(9, 6))
importances.plot(kind='barh', color='steelblue')
plt.xlabel('Feature Importance')
plt.title('Random Forest – Top 15 Feature Importances')
plt.gca().invert_yaxis()
plt.tight_layout(); plt.show()

## 2. Effect of Number of Trees

In [ ]:
n_trees = [1, 5, 10, 20, 50, 100, 200]
test_accs = []
for n in n_trees:
    rf_n = RandomForestClassifier(n_estimators=n, random_state=42)
    rf_n.fit(X_train, y_train)
    test_accs.append(accuracy_score(y_test, rf_n.predict(X_test)))

plt.figure(figsize=(8, 4))
plt.plot(n_trees, test_accs, marker='o', color='steelblue')
plt.xlabel('Number of Trees'); plt.ylabel('Test Accuracy')
plt.title('Random Forest – Accuracy vs. Number of Trees')
plt.xscale('log'); plt.tight_layout(); plt.show()

## 🏋️ Exercises

1. Tune `max_features` in `RandomForestClassifier` (`'sqrt'`, `'log2'`, `None`). Which works best?
2. Add `XGBClassifier` from the `xgboost` library and compare it to `GradientBoostingClassifier`.
3. Use `permutation_importance` (from `sklearn.inspection`) and compare with built-in feature importances.

---
**Next ▶ [Episode 15 – Support Vector Machines](05_svm.ipynb)**